In [1]:
from  ..message import *

load_dotenv(override=True)
DEEPSEEK_API_KEY=os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL=os.getenv('DEEPSEEK_BASE_URL')

model=init_chat_model(
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek-v4-flash',
    model_provider='deepseek',
)

def keep_recent_messages(messages, max_pairs=3):
    """
    保留最近的 N 轮对话（每轮 = user + assistant 或 Human + AI）。
    支持字典格式和 LangChain BaseMessage 格式。
    返回与输入相同类型的列表。
    """
    if not messages:
        return []

    # 检测类型：如果第一个元素是 BaseMessage 的子类，则按 BaseMessage 处理
    is_base = isinstance(messages[0], BaseMessage)

    if is_base:
        # 处理 BaseMessage
        system_msgs = [m for m in messages if isinstance(m, SystemMessage)]
        conversation_msgs = [m for m in messages if not isinstance(m, SystemMessage)]
        recent_msgs = conversation_msgs[-(max_pairs * 2):]
        return system_msgs + recent_msgs
    else:
        # 处理字典格式
        system_msgs = [m for m in messages if m.get("role") == "system"]
        conversation_msgs = [m for m in messages if m.get("role") != "system"]
        recent_msgs = conversation_msgs[-(max_pairs * 2):]
        return system_msgs + recent_msgs
def add_messages(messages_log,messages,model):
    messages_log.append(messages)
    response=model.invoke(messages_log)
    rprint(response.content)
    messages_log.append(response)
    return messages_log

In [2]:
conversation_log=[SystemMessage('你是一个诗人')]
message1=HumanMessage('帮我写一首描写荷花的七言诗')
conversation_log=add_messages(conversation_log,message1,model)
message2=HumanMessage('将这首诗的第一句翻译为英文')
conversation_log=add_messages(conversation_log,message2,model)
message3=HumanMessage('将这首诗的第二句翻译为英文')
conversation_log=add_messages(conversation_log,message3,model)
print('=======目前对话包含===========')
for msg in conversation_log:
    print(f"{msg.type}: {msg.content}")
n=2
print('=======保留最近n轮对话后包含===========')
filterMsg=keep_recent_messages(conversation_log,n)
for msg in filterMsg:
    print(f"{msg.type}: {msg.content}")

《荷塘》
碧波荡漾映荷塘，翠盖摇风送暗香。
玉立娉婷羞月色，淤泥不染自清扬。

The jade-green ripples spread, reflecting the lotus pond.

Here is the translation of the second line:

"Green canopies sway in the breeze, sending faint fragrance afar."

=======目前对话包含===========
system: 你是一个诗人
human: 帮我写一首描写荷花的七言诗
ai: 《荷塘》
碧波荡漾映荷塘，翠盖摇风送暗香。
玉立娉婷羞月色，淤泥不染自清扬。
human: 将这首诗的第一句翻译为英文
ai: The jade-green ripples spread, reflecting the lotus pond.
human: 将这首诗的第二句翻译为英文
ai: Here is the translation of the second line:

"Green canopies sway in the breeze, sending faint fragrance afar."
=======保留最近n轮对话后包含===========
system: 你是一个诗人
human: 将这首诗的第一句翻译为英文
ai: The jade-green ripples spread, reflecting the lotus pond.
human: 将这首诗的第二句翻译为英文
ai: Here is the translation of the second line:

"Green canopies sway in the breeze, sending faint fragrance afar."
